# build_features_extra.ipynb — mở rộng gold/features (p22, p20, p16, p24, p09)

Notebook này **đọc trực tiếp code thật** của các pipeline liên quan (`pipelines/p22`, `p20`, `p16`, `p24`, `p09`,
và `notebooks/python/build_features.py`) trước khi viết, để join đúng schema/khóa thay vì đoán.

**Việc làm theo đúng thứ tự đã thống nhất:**
1. p22 (odds handicap/totals) + p20 (wiki/news) — nhanh, rủi ro thấp
2. p16 (injuries) — as-of join qua thời gian
3. p24 (af_match_events/lineups/stats) — bắc cầu match_id trước
4. p09 (fdo_matches) — đối chiếu/vá lỗ hổng p03, KHÔNG tạo feature riêng

**2 điều chỉnh so với đánh giá lúc trước (đọc code thật mới phát hiện):**
- **p22**: `match_id` của The Odds API là ID riêng của họ, khác hẳn `E0_...` của `fd_matches` → **không thể
  join theo `match_id`** như đã nói ban đầu, phải join theo `(home_key, away_key, ngày)`. May mắn là
  `seed/team_alias.csv` đã có sẵn `source=odds` khớp đúng tên đội của Odds API rồi.
- **p20**: `silver/text/wiki_articles/entity=team` **không phải** time-series như `wm_pageviews` — mỗi dòng là
  1 bài Wikipedia full-text (gần như không đổi qua các lần cào) → không hợp làm feature rolling theo trận,
  chỉ hợp làm bảng mô tả (dim). Còn `google_news_articles` chỉ 2/7 query gắn thẳng 1 CLB → phải quét chuỗi
  `text` tìm tên đội thay vì dựa vào cột `query`.

**Lưu ý về rò rỉ dữ liệu (leakage):** mọi join bên dưới đều dùng `< match_date` hoặc as-of
`<= match_date` — không dùng dữ liệu tương lai, theo đúng nguyên tắc `build_features.py` đang áp dụng.


In [ ]:
# [DÀNH RIÊNG CHO GOOGLE COLAB] — bỏ qua nếu chạy trong container Airflow đã có sẵn code
import sys, os
os.environ['MINIO_ENDPOINT'] = 'http://20.41.113.183:9000'
os.environ['MINIO_ACCESS_KEY'] = 'minioadmin'
os.environ['MINIO_SECRET_KEY'] = 'minioadmin123'
if 'SEED_PATH' in os.environ: del os.environ['SEED_PATH']
if 'google.colab' in sys.modules:
    !rm -rf /content/Lab
    !git clone -b feat-e https://github.com/nbngoc123/Lab.git /content/Lab
    os.chdir('/content/Lab/football-lake')
    sys.path.insert(0, '/content/Lab/football-lake')
    !pip install minio duckdb pandas boto3 python-dotenv
    print('Setup xong Colab!')


## 0. Tái dùng `build_features.py` thay vì viết lại

`notebooks/python/build_features.py` đã có sẵn `LocalStore`/`MinioStore`, `load_alias`, `map_team`,
`win()`, `first_present`, `num` — import lại để không lệch quy ước (team_key, khóa alias, cách đọc silver...).


In [ ]:
import sys, os, io
from pathlib import Path
import duckdb
import pandas as pd

sys.path.insert(0, str(Path.cwd() / "notebooks" / "python"))
import build_features as bf

store = bf.LocalStore(os.environ["ROOT"]) if os.environ.get("ROOT") else bf.MinioStore()
alias = bf.load_alias(bf.find_seed())
DIVISION = "E0"

def read_gold(store, key):
    """Đọc 1 file gold/features/*.parquet cụ thể (khác silver: không phải prefix nhiều part)."""
    if isinstance(store, bf.LocalStore):
        return pd.read_parquet(store.root / key)
    return pd.read_parquet(io.BytesIO(store.mio.read_bytes(key)))

print("[spine] đọc lại fd_matches làm trục chính -- giống hệt build_features.py")
m = bf.load_matches(store, alias, DIVISION)
con = duckdb.connect()
con.register("m", m)
bf.build_team_match(con)   # tạo bảng 'tm' (1 dòng/đội/trận) -- dùng lại cho as-of join bên dưới
print(f"spine: {len(m):,} trận, {m.match_date.min()} -> {m.match_date.max()}")


## 1. p22 — Odds handicap (spreads) & totals (over/under)

Nguồn: `pipelines/p22/main.py` → `silver/betting/odds_h2h|odds_spreads|odds_totals`.
`match_id` là ID của The Odds API, không trùng `fd_matches` → join theo `(home_key, away_key, ngày)`.

⚠️ **Giới hạn dữ liệu**: bản free của The Odds API chỉ trả kèo các trận **sắp diễn ra**, không có kèo lịch
sử. Nếu bạn mới chạy p22 vài ngày gần đây thì coverage của `feature_market_ah` trên tập train sẽ THẤP —
đây không phải lỗi join, cứ để pipeline chạy hàng ngày, coverage sẽ tăng dần cho các trận tương lai.


In [ ]:
def _empty(cols):
    return pd.DataFrame(columns=cols)

def load_odds_handicap(store, alias):
    sp = store.read_prefix("silver/betting/odds_spreads/")
    if sp.empty:
        print("  ! không có odds_spreads -> ah_* sẽ NULL")
        sp = _empty(["home_key","away_key","match_date","side_key","price","point","bookmaker"])
    else:
        sp["home_key"] = bf.map_team(sp["home_team"], "odds", alias, "odds_spreads.home_team")
        sp["away_key"] = bf.map_team(sp["away_team"], "odds", alias, "odds_spreads.away_team")
        sp["side_key"] = bf.map_team(sp["team"], "odds", alias, "odds_spreads.team")
        sp["match_date"] = pd.to_datetime(sp["commence_time"], errors="coerce").dt.date

    tt = store.read_prefix("silver/betting/odds_totals/")
    if tt.empty:
        print("  ! không có odds_totals -> ou_* sẽ NULL")
        tt = _empty(["home_key","away_key","match_date","name","price","point","bookmaker"])
    else:
        tt["home_key"] = bf.map_team(tt["home_team"], "odds", alias, "odds_totals.home_team")
        tt["away_key"] = bf.map_team(tt["away_team"], "odds", alias, "odds_totals.away_team")
        tt["match_date"] = pd.to_datetime(tt["commence_time"], errors="coerce").dt.date
    return sp, tt

ah_spreads, ah_totals = load_odds_handicap(store, alias)
con.register("ah_spreads", ah_spreads)
con.register("ah_totals", ah_totals)

con.execute("""
CREATE OR REPLACE TABLE feature_market_ah AS
WITH sp AS (
  SELECT home_key, away_key, match_date,
         AVG(price) FILTER (WHERE side_key = home_key) AS ah_home_price_avg,
         AVG(price) FILTER (WHERE side_key = away_key) AS ah_away_price_avg,
         AVG(point) FILTER (WHERE side_key = home_key) AS ah_home_point_avg,
         COUNT(DISTINCT bookmaker) AS ah_n_books
  FROM ah_spreads GROUP BY 1, 2, 3
),
tt AS (
  SELECT home_key, away_key, match_date,
         AVG(price) FILTER (WHERE name = 'Over')  AS ou_over_price_avg,
         AVG(price) FILTER (WHERE name = 'Under') AS ou_under_price_avg,
         AVG(point) AS ou_line_avg,
         COUNT(DISTINCT bookmaker) AS ou_n_books
  FROM ah_totals GROUP BY 1, 2, 3
)
SELECT m.match_id,
       sp.ah_home_price_avg, sp.ah_away_price_avg, sp.ah_home_point_avg, sp.ah_n_books,
       tt.ou_over_price_avg, tt.ou_under_price_avg, tt.ou_line_avg, tt.ou_n_books
FROM m
LEFT JOIN sp ON sp.home_key = m.home_key AND sp.away_key = m.away_key AND sp.match_date = m.match_date
LEFT JOIN tt ON tt.home_key = m.home_key AND tt.away_key = m.away_key AND tt.match_date = m.match_date
""")

cov = con.execute("SELECT round(avg((ah_n_books IS NOT NULL)::INT),3) AS ah_cov, "
                   "round(avg((ou_n_books IS NOT NULL)::INT),3) AS ou_cov FROM feature_market_ah").df()
print("feature_market_ah coverage (tỉ lệ trận có kèo handicap/totals):")
print(cov.to_string(index=False))


## 2. p20 — Wikipedia & Google News

- `wiki_articles/entity=team`: **không** đưa vào feature rolling (giải thích ở đầu file) — chỉ liệt kê để
  bạn dùng cho OBT/mô tả đội nếu cần.
- `google_news_articles`: quét cột `text` (title+summary) tìm bất kỳ tên đội nào có trong
  `seed/team_alias.csv` (không chỉ dựa cột `query`) → đếm số bài nhắc tới mỗi đội theo ngày, rồi cộng dồn
  7 ngày / 28 ngày trước trận (giống hệt cách `build_attention()` đang làm với `wm_pageviews`).


In [ ]:
# --- wiki_articles: chỉ xem qua, KHÔNG join vào feature_match_ml (dữ liệu tĩnh, không đổi theo trận) ---
wiki = store.read_prefix("silver/text/wiki_articles/entity=team/")
if not wiki.empty:
    dim_team_wiki = (wiki.sort_values("_key")
                          .drop_duplicates(["title","lang"], keep="last")
                          [["title","lang","page_id","word_count","fetched_date"]])
    print(f"dim_team_wiki_profile: {len(dim_team_wiki)} dòng (tĩnh, hợp cho OBT chứ không phải feature ML)")
else:
    dim_team_wiki = pd.DataFrame()
    print("  ! không có wiki_articles/entity=team")


In [ ]:
def load_news_buzz(store, alias):
    n = store.read_prefix("silver/text/google_news_articles/")
    if n.empty:
        print("  ! không có google_news_articles -> feature_team_news_buzz rỗng")
        return pd.DataFrame(columns=["team_key", "date", "n_mentions"])
    n["date"] = pd.to_datetime(n["published_ts"], errors="coerce", utc=True).dt.tz_localize(None).dt.date
    n = n.dropna(subset=["date", "text"])

    _, any_src = alias
    # alias dài xét trước để tránh khớp nhầm (vd 'Man' khớp cả Man City lẫn Man United)
    aliases = sorted(any_src.items(), key=lambda kv: -len(kv[0]))

    rows = []
    for r in n.itertuples():
        low = r.text.lower()
        hit = {tk for al, tk in aliases if al in low}
        for tk in hit:
            rows.append((tk, r.date))
    out = pd.DataFrame(rows, columns=["team_key", "date"])
    if out.empty:
        return pd.DataFrame(columns=["team_key", "date", "n_mentions"])
    out = out.groupby(["team_key", "date"]).size().reset_index(name="n_mentions")
    print(f"  · news_buzz: quét {len(n):,} bài -> {len(out):,} dòng team-ngày, "
          f"{out.team_key.nunique()} đội được nhắc tới")
    return out

nb_daily = load_news_buzz(store, alias)
con.register("nb", nb_daily)

con.execute("""
CREATE OR REPLACE TABLE feature_team_news_buzz AS
SELECT tm.match_id, tm.team_key,
  SUM(nb.n_mentions) FILTER (WHERE nb.date >= tm.match_date - 7  AND nb.date < tm.match_date) AS news_n7,
  SUM(nb.n_mentions) FILTER (WHERE nb.date >= tm.match_date - 28 AND nb.date < tm.match_date) AS news_n28
FROM tm LEFT JOIN nb ON nb.team_key = tm.team_key
GROUP BY tm.match_id, tm.team_key
""")
cov = con.execute("SELECT round(avg((news_n28 IS NOT NULL AND news_n28 > 0)::INT),3) AS news_cov "
                   "FROM feature_team_news_buzz").df()
print("feature_team_news_buzz coverage:", cov.iloc[0,0])


## 3. p16 — Injuries (as-of join)

Nguồn: `pipelines/p16/main.py` → `silver/players/pr_player_injuries` (team, player_name, injury_type,
`ingest_date`) và `silver/dim/pr_club_injury_summary` (team, total_injuries, `ingest_date`).

Đây là **snapshot theo ngày cào**, không có `match_id` → dùng **ASOF JOIN** của DuckDB: mỗi (team, trận)
lấy snapshot chấn thương **gần nhất nhưng trước** `match_date` (không rò rỉ tương lai).

⚠️ Nếu p16 mới chạy được vài lần, số `ingest_date` sẽ ít → các trận **trước** lần scrape đầu tiên sẽ NULL
(đúng theo thiết kế, không phải lỗi), còn các trận sau đó dùng chung 1 snapshot cho tới lần scrape kế tiếp.


In [ ]:
def load_injuries(store, alias):
    p = store.read_prefix("silver/players/pr_player_injuries/")
    s = store.read_prefix("silver/dim/pr_club_injury_summary/")
    cols = ["team_key", "ingest_date", "n_injured_players", "total_injuries"]
    if p.empty:
        print("  ! không có pr_player_injuries -> feature_team_injuries rỗng")
        return pd.DataFrame(columns=cols)
    p["team_key"] = bf.map_team(p["team"], "physioroom", alias, "pr_player_injuries.team")
    p["ingest_date"] = pd.to_datetime(p["ingest_date"], errors="coerce").dt.date
    agg = p.groupby(["team_key", "ingest_date"]).size().reset_index(name="n_injured_players")
    if not s.empty:
        s["team_key"] = bf.map_team(s["team"], "physioroom", alias, "pr_club_injury_summary.team")
        s["ingest_date"] = pd.to_datetime(s["ingest_date"], errors="coerce").dt.date
        agg = agg.merge(s[["team_key", "ingest_date", "total_injuries"]],
                         on=["team_key", "ingest_date"], how="outer")
    print(f"  · injuries: {agg['ingest_date'].nunique()} lần scrape, {agg['team_key'].nunique()} đội")
    return agg

inj = load_injuries(store, alias).sort_values(["team_key", "ingest_date"])
con.register("inj", inj)

con.execute("""
CREATE OR REPLACE TABLE feature_team_injuries AS
SELECT tm.match_id, tm.team_key, inj.ingest_date AS injury_asof_date,
       inj.n_injured_players, inj.total_injuries
FROM tm
ASOF LEFT JOIN inj
  ON tm.team_key = inj.team_key AND tm.match_date >= inj.ingest_date
""")
cov = con.execute("SELECT round(avg((n_injured_players IS NOT NULL)::INT),3) AS inj_cov "
                   "FROM feature_team_injuries").df()
print("feature_team_injuries coverage:", cov.iloc[0,0],
      "(thấp là bình thường nếu p16 mới chạy gần đây)")


## 4. p24 — Bắc cầu `match_id` (fd_matches ↔ af_fixtures)

Nguồn: `pipelines/p24/main.py` → `silver/matches/af_fixtures` (khóa `fixture_id`, tên đội theo
API-Football — đã có sẵn alias `source=apifootball` trong `seed/team_alias.csv`).

`af_match_events` / `af_lineups` / `af_match_stats` đều khóa theo `fixture_id`, không theo `E0_...` →
bắc cầu bằng `(home_key, away_key, ngày lệch tối đa 1 ngày)` để phòng lệch múi giờ. **Bước này chỉ tạo
bảng ánh xạ + demo 1 feature từ `af_match_stats`** — việc khai thác đầy đủ `af_match_events`/`af_lineups`
(thẻ theo phút, thay người...) nên làm ở notebook riêng SAU KHI coverage bảng ánh xạ đủ tốt.


In [ ]:
def build_af_fd_bridge(store, alias, con):
    af = store.read_prefix("silver/matches/af_fixtures/")
    if af.empty:
        print("  ! không có af_fixtures -> không bắc cầu được")
        return pd.DataFrame(columns=["match_id","fixture_id","fd_date","af_date","score_match"])
    af = af.sort_values("_key").drop_duplicates("fixture_id", keep="last").copy()
    af["home_key"] = bf.map_team(af["home_team"], "apifootball", alias, "af_fixtures.home_team")
    af["away_key"] = bf.map_team(af["away_team"], "apifootball", alias, "af_fixtures.away_team")
    dt = pd.to_datetime(af["date"], errors="coerce", utc=True)
    af["match_date"] = dt.dt.tz_localize(None).dt.date
    con.register("af_fx", af[["fixture_id","home_key","away_key","match_date","home_goals","away_goals"]])
    bridge = con.execute("""
        SELECT m.match_id, af.fixture_id, m.match_date AS fd_date, af.match_date AS af_date,
               (m.home_goals = af.home_goals AND m.away_goals = af.away_goals) AS score_match
        FROM m JOIN af_fx af
          ON af.home_key = m.home_key AND af.away_key = m.away_key
         AND abs(date_diff('day', af.match_date, m.match_date)) <= 1
    """).df()
    n_m = con.execute("SELECT count(*) FROM m").fetchone()[0]
    print(f"  · bridge fd<->af: khớp {len(bridge):,}/{n_m:,} trận ({len(bridge)/max(n_m,1):.0%}); "
          f"tỉ số trùng khớp {bridge['score_match'].mean():.0%}" if len(bridge) else "  · bridge rỗng")
    return bridge

bridge_fd_af = build_af_fd_bridge(store, alias, con)

# demo: 1 feature từ af_match_stats để chứng minh bridge dùng được (không ghi vào feature_match_ml_v2)
stats = store.read_prefix("silver/matches/af_match_stats/")
if not stats.empty and not bridge_fd_af.empty:
    poss_col = bf.first_present(stats, ["ball_possession", "possession"])
    if poss_col:
        con.register("af_stats", stats[["fixture_id", "team", poss_col]])
        con.register("bridge", bridge_fd_af)
        demo = con.execute(f"""
            SELECT b.match_id, s.team, s.{poss_col} AS possession
            FROM bridge b JOIN af_stats s ON s.fixture_id = b.fixture_id
            LIMIT 10
        """).df()
        print(demo.to_string(index=False))
    else:
        print("  ! af_match_stats không có cột possession -- kiểm tra lại tên cột thật")
else:
    print("  ! chưa đủ dữ liệu (af_match_stats hoặc bridge rỗng) để demo feature")


## 5. p09 — Đối chiếu với `fd_matches` (KHÔNG tạo feature riêng)

Theo đúng thống nhất: p09 (football-data.org) dùng để **vá lỗ hổng / kiểm tra chéo** dữ liệu của p03, không
tạo bảng feature riêng (tránh double-count 1 nguồn 2 lần trong model).


In [ ]:
fdo = store.read_prefix("silver/matches/fdo_matches/")
if not fdo.empty:
    fdo = fdo.copy()
    fdo["home_key"] = bf.map_team(fdo["home_team"], "fdo", alias, "fdo_matches.home_team")
    fdo["away_key"] = bf.map_team(fdo["away_team"], "fdo", alias, "fdo_matches.away_team")
    fdo["match_date"] = pd.to_datetime(fdo["utc_date"], errors="coerce").dt.date
    con.register("fdo", fdo[["match_id","home_key","away_key","match_date",
                              "home_goals","away_goals","status"]])

    gap = con.execute("""
        SELECT fdo.match_id, fdo.home_key, fdo.away_key, fdo.match_date,
               fdo.home_goals, fdo.away_goals
        FROM fdo
        LEFT JOIN m ON m.home_key = fdo.home_key AND m.away_key = fdo.away_key
                    AND abs(date_diff('day', m.match_date, fdo.match_date)) <= 1
        WHERE m.match_id IS NULL AND fdo.status = 'FINISHED'
    """).df()

    mismatch = con.execute("""
        SELECT m.match_id, m.match_date, m.home_goals, m.away_goals,
               fdo.home_goals AS fdo_home_goals, fdo.away_goals AS fdo_away_goals
        FROM m JOIN fdo ON fdo.home_key = m.home_key AND fdo.away_key = m.away_key
                       AND abs(date_diff('day', m.match_date, fdo.match_date)) <= 1
        WHERE m.home_goals != fdo.home_goals OR m.away_goals != fdo.away_goals
    """).df()

    print(f"fdo_matches: {len(fdo):,} trận.")
    print(f"  · {len(gap)} trận fdo CÓ nhưng fd_matches KHÔNG có (ứng viên vá lỗ hổng cho p03).")
    print(f"  · {len(mismatch)} trận tỉ số LỆCH giữa 2 nguồn (cần soát lại thủ công).")
    if len(mismatch): print(mismatch.to_string(index=False))
else:
    print("  ! không có silver/matches/fdo_matches -> chạy p09 trước")


## 6. Ghi kết quả + hợp nhất vào `feature_match_ml_v2`

Ghi các bảng mới vào `gold/features/` **dưới tên riêng** (không đè `feature_market`/`feature_match_ml` gốc),
sau đó tạo thêm `feature_match_ml_v2.parquet` = `feature_match_ml` gốc LEFT JOIN 3 nhóm mới
(`feature_market_ah`, `feature_team_news_buzz`, `feature_team_injuries`). `bridge_fd_af_match` ghi riêng để
dùng cho notebook khai thác `af_match_events`/`af_lineups` sau này.


In [ ]:
new_tables = {
    "feature_market_ah": con.execute("SELECT * FROM feature_market_ah").df(),
    "feature_team_news_buzz": con.execute("SELECT * FROM feature_team_news_buzz").df(),
    "feature_team_injuries": con.execute("SELECT * FROM feature_team_injuries").df(),
}
for name, df in new_tables.items():
    store.write_parquet(f"gold/features/{name}.parquet", df)
store.write_parquet("gold/features/bridge_fd_af_match.parquet", bridge_fd_af)
if not dim_team_wiki.empty:
    store.write_parquet("gold/dim/dim_team_wiki_profile.parquet", dim_team_wiki)


In [ ]:
ml0 = read_gold(store, "gold/features/feature_match_ml.parquet")
con.register("ml0", ml0)
con.register("ah", new_tables["feature_market_ah"])
con.register("nb", new_tables["feature_team_news_buzz"])
con.register("inj", new_tables["feature_team_injuries"])

ml_v2 = con.execute("""
SELECT ml0.*,
  ah.ah_home_price_avg, ah.ah_away_price_avg, ah.ah_home_point_avg, ah.ah_n_books,
  ah.ou_over_price_avg, ah.ou_under_price_avg, ah.ou_line_avg, ah.ou_n_books,
  nbh.news_n7  AS home_news_n7,  nbh.news_n28 AS home_news_n28,
  nba.news_n7  AS away_news_n7,  nba.news_n28 AS away_news_n28,
  (nbh.news_n28 - nba.news_n28)  AS diff_news_n28,
  injh.n_injured_players AS home_n_injured, inja.n_injured_players AS away_n_injured,
  (injh.n_injured_players - inja.n_injured_players) AS diff_n_injured
FROM ml0
LEFT JOIN ah  ON ah.match_id = ml0.match_id
LEFT JOIN nb  nbh ON nbh.match_id = ml0.match_id AND nbh.team_key = ml0.home_key
LEFT JOIN nb  nba ON nba.match_id = ml0.match_id AND nba.team_key = ml0.away_key
LEFT JOIN inj injh ON injh.match_id = ml0.match_id AND injh.team_key = ml0.home_key
LEFT JOIN inj inja ON inja.match_id = ml0.match_id AND inja.team_key = ml0.away_key
""").df()

assert len(ml_v2) == len(ml0), "feature_match_ml_v2 lệch số dòng so với bản gốc -- kiểm tra lại join"
store.write_parquet("gold/features/feature_match_ml_v2.parquet", ml_v2)

print(f"feature_match_ml_v2: {len(ml_v2):,} dòng x {ml_v2.shape[1]} cột "
      f"(gốc: {ml0.shape[1]} cột, +{ml_v2.shape[1]-ml0.shape[1]} cột mới)")
print("\nĐộ phủ các nhóm feature mới (toàn bộ trận, kể cả trận cũ chưa có dữ liệu p22/p20/p16):")
print(con.execute("""
    SELECT round(avg((ah_n_books IS NOT NULL)::INT),3)      AS odds_handicap,
           round(avg((home_news_n28 IS NOT NULL)::INT),3)   AS news_buzz,
           round(avg((home_n_injured IS NOT NULL)::INT),3)  AS injuries
    FROM ml_v2
""").df().to_string(index=False))


## Bước tiếp theo gợi ý

- Chạy p22/p20/p16/p24/p09 đều đặn vài ngày để tích lũy nhiều `ingest_date` hơn → coverage của
  `feature_market_ah` / `feature_team_news_buzz` / `feature_team_injuries` sẽ tăng dần (đặc biệt injuries
  và odds handicap hiện chỉ có 1 snapshot).
- Nếu `map_team(...)` in cảnh báo tên chưa có trong alias, thêm dòng
  `source,<tên gốc>,<team_key>` vào `seed/team_alias.csv` (nguồn `physioroom`, `fdo` hiện đang dùng
  fallback `any_src`, nên rà lại danh sách cảnh báo khi chạy thật với dữ liệu MinIO đầy đủ).
- `bridge_fd_af_match.parquet` là nền cho notebook tiếp theo: dùng `af_match_events`/`af_lineups` để làm
  feature thẻ phạt/thay người theo phút — hiện mới demo 1 cột `possession` từ `af_match_stats`.
